# 8onzas unified agent — availability evals (lab A2)

Same agent as menu (`lab/agent_v1.py`): **two-phase** (tools then structured pack).

- Phase 1: `check_availability` only (ReAct)
- Phase 2: `AgentResponse` via `with_structured_output` (no tools — Groq-safe)

This notebook focuses on **availability** metrics.

Requires `GROQ_API_KEY` in `.env` (fallback: `OPENROUTER_API_KEY`). Do not commit secrets.

In [1]:
import sys
from pathlib import Path

LAB_DIR = Path.cwd()
if LAB_DIR.name != "lab":
    candidate = LAB_DIR / "lab"
    if candidate.exists():
        sys.path.insert(0, str(candidate))
else:
    sys.path.insert(0, str(LAB_DIR))

from agent_v1 import (
    AgentResponse,
    build_agent,
    compute_availability,
    evaluate_availability,
    invoke_agent,
    load_availability_cases,
    load_calendar,
    load_dotenv,
    load_menu,
    load_profile,
    resolve_groq_api_key,
)

load_dotenv()
resolve_groq_api_key()
profile = load_profile()
menu = load_menu()
calendar = load_calendar()
suite = load_availability_cases()
profile.name, calendar.capacity, len(suite.cases)

('8onzas', {'table': 8, 'terrace': 4, 'birthday': 12}, 4)

## Calendar + pure tool check (no LLM)

In [2]:
print("bookings", len(calendar.bookings))
for case in suite.cases:
    exp = case.expect
    pure = compute_availability(
        date=exp.date,
        time=exp.time,
        party_size=exp.party_size,
        space=exp.space or "table",
        calendar=calendar,
        profile=profile,
    )
    print(case.id, "expect", exp.available, "pure", pure["available"], pure.get("reason"))

bookings 4
avail-free-slot expect True pure True None
avail-busy-slot expect False pure False sin capacidad en ese slot
avail-closed-monday expect False pure False cerrado (monday)
avail-party-too-big expect False pure False party_size 20 fuera de rango (1-12)


## One agent — smoke menu + availability

In [3]:
agent = build_agent(profile, menu, calendar)
menu_smoke = invoke_agent("¿Cuánto sale el latte?", agent=agent)
free = invoke_agent(
    "¿Hay mesa para 2 el viernes 27/11/2026 a las 19:00?",
    agent=agent,
)
busy = invoke_agent(
    "¿Hay mesa para 4 el sábado 28/11/2026 a las 20:00?",
    agent=agent,
)
assert all(isinstance(r, AgentResponse) for r in (menu_smoke, free, busy))
menu_smoke.model_dump(), free.model_dump(), busy.model_dump()

OpenAIInvalidRequestError: Error code: 400 - json mode cannot be combined with tool/function calling

## Evaluate availability suite

In [4]:
metrics = evaluate_availability(suite, agent=agent)
print(f"availability_accuracy={metrics['availability_accuracy']}")
print(f"slot_match={metrics['slot_match']}")
print(f"tool_discipline={metrics['tool_discipline']}")
metrics["rows"]

KeyboardInterrupt: 

## Notes

- Canonical: `lab/agent_v1.py` — Phase 1 ReAct + Phase 2 structured pack (`AgentBundle`).
- Invalid tool args → ToolMessage → fix in phase 1; structured failures → one repair retry then visible error.
- Menu P/R: `01_menu_agent_v1.ipynb` on the same `build_agent`.
- No soft-reserve yet (A3/A4/B3).